In [5]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("scikit-fingerprints/MoleculeNet_BBBP")
print(ds)

df = ds["train"].to_pandas()
print(df.shape)
print(df.columns.tolist())
print(df.head())

df.to_csv("../data/bbbp.csv", index=False)

DatasetDict({
    train: Dataset({
        features: ['SMILES', 'label'],
        num_rows: 2039
    })
})
(2039, 2)
['SMILES', 'label']
                                              SMILES  label
0                   [Cl].CC(C)NCC(O)COc1cccc2ccccc12      1
1           C(=O)(OC(C)(C)C)CCCc1ccc(cc1)N(CCCl)CCCl      1
2  c12c3c(N4CCN(C)CC4)c(F)cc1c(c(C(O)=O)cn2C(C)CO...      1
3                   C1CCN(CC1)Cc1cccc(c1)OCCCNC(=O)C      1
4  Cc1onc(c2ccccc2Cl)c1C(=O)N[C@H]3[C@H]4SC(C)(C)...      1


In [6]:
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.*")

df = pd.read_csv("../data/bbbp.csv")

# 1. Class balance
print("--- Class balance ---")
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True).round(3))

# 2. Parse SMILES
df["mol"] = df["SMILES"].apply(Chem.MolFromSmiles)
print("\nUnparseable SMILES:", df["mol"].isna().sum())
print("Duplicate SMILES:", df["SMILES"].duplicated().sum())
print("Multi-fragment SMILES (salts):", df["SMILES"].str.contains(r"\.").sum())

# 3. Clean: drop unparseable, keep largest fragment
def largest_fragment(mol):
    frags = Chem.GetMolFrags(mol, asMols=True)
    return max(frags, key=lambda m: m.GetNumHeavyAtoms())

df = df.dropna(subset=["mol"]).copy()
df["mol"] = df["mol"].apply(largest_fragment)
df["clean_smiles"] = df["mol"].apply(Chem.MolToSmiles)

# 4. Conflicting labels for the same molecule
conflicts = df.groupby("clean_smiles")["label"].nunique()
print("\nSame molecule, conflicting labels:", (conflicts > 1).sum())

# 5. Scaffolds
df["scaffold"] = df["clean_smiles"].apply(
    lambda s: MurckoScaffold.MurckoScaffoldSmiles(smiles=s, includeChirality=False)
)
print("Unique scaffolds:", df["scaffold"].nunique(), "of", len(df), "molecules")

--- Class balance ---
label
1    1560
0     479
Name: count, dtype: int64
label
1    0.765
0    0.235
Name: proportion, dtype: float64

Unparseable SMILES: 0
Duplicate SMILES: 0
Multi-fragment SMILES (salts): 105

Same molecule, conflicting labels: 11
Unique scaffolds: 1021 of 2039 molecules


In [7]:
import sys 
sys.path.append("..")

from src.data import load_clean 
from src.split import scaffold_split 

df = load_clean()
train_idx, valid_idx, test_idx = scaffold_split(df)

print("Sizes:", len(train_idx), len(valid_idx), len(test_idx))

# no scaffold should appear in more than one split
s_tr = set(df.loc[train_idx, "scaffold"])
s_va = set(df.loc[valid_idx, "scaffold"])
s_te = set(df.loc[test_idx, "scaffold"])
print("Scaffold overlap train/test:", len(s_tr & s_te),
      "| train/valid:", len(s_tr & s_va), "| valid/test:", len(s_va & s_te))

for name, idx in [("train", train_idx), ("valid", valid_idx), ("test", test_idx)]:
    print(name, "fraction permeable:", round(df.loc[idx, "label"].mean(), 3))

Start: 2039 | removed conflicting-label rows: 22 | removed duplicates: 62 | final: 1955
Sizes: 1564 195 196
Scaffold overlap train/test: 0 | train/valid: 0 | valid/test: 0
train fraction permeable: 0.826
valid fraction permeable: 0.497
test fraction permeable: 0.536
